# Usage of PySpark SQL

In [1]:
from pyspark.sql import SparkSession

spark = (SparkSession.builder
                     .appName("Analyzing an unknown article.")
                     .getOrCreate())


In [2]:
## documentation
spark.read??

In [9]:
file_path = r'music_news.txt' # fill in

In [10]:
article = spark.read.text(file_path)

In [11]:
article

DataFrame[value: string]

In [17]:

article.select(article.value)
article.select(article['value'])
article.select(col('value'))
article.select('value')

DataFrame[value: string]

In [12]:
article.show(5, truncate=False)

+------------------------------------------------------------------------------+
|value                                                                         |
+------------------------------------------------------------------------------+
|All Songs Considered: Peter Gabriel goes pop, Olivia Rodrigo might 'drop dead'|
|Young pop stars are burning out — and singing through it                      |
|Infinity Song: Tiny Desk Concert                                              |
|Anne Hathaway is a holy pop star in 'Mother Mary'                             |
|Musician Jesse Welles sings about the news, to great fanfare                  |
+------------------------------------------------------------------------------+
only showing top 5 rows


In [14]:
from pyspark.sql.functions import col, split
lines = article.select(split(col('value'), ' ').alias('lines'))
lines.show(5, truncate=False)

+-------------------------------------------------------------------------------------------+
|lines                                                                                      |
+-------------------------------------------------------------------------------------------+
|[All, Songs, Considered:, Peter, Gabriel, goes, pop,, Olivia, Rodrigo, might, 'drop, dead']|
|[Young, pop, stars, are, burning, out, —, and, singing, through, it]                       |
|[Infinity, Song:, Tiny, Desk, Concert]                                                     |
|[Anne, Hathaway, is, a, holy, pop, star, in, 'Mother, Mary']                               |
|[Musician, Jesse, Welles, sings, about, the, news,, to, great, fanfare]                    |
+-------------------------------------------------------------------------------------------+
only showing top 5 rows


In [15]:
from pyspark.sql.functions import explode
words = lines.select(explode(col('lines')).alias('word'))
words.show(5, truncate=False)

+-----------+
|word       |
+-----------+
|All        |
|Songs      |
|Considered:|
|Peter      |
|Gabriel    |
+-----------+
only showing top 5 rows


In [16]:
from pyspark.sql.functions import lower
words_lower = words.select(lower(col('word')).alias('word'))
words_lower.show(5, truncate=False)

+-----------+
|word       |
+-----------+
|all        |
|songs      |
|considered:|
|peter      |
|gabriel    |
+-----------+
only showing top 5 rows


In [20]:
from pyspark.sql.functions import regexp_extract
words_clean = words_lower.select(regexp_extract(col('word'), r'(\W+)?([a-z]+)', 2).alias('word'))
words_clean.show(5, truncate=False)

+----------+
|word      |
+----------+
|all       |
|songs     |
|considered|
|peter     |
|gabriel   |
+----------+
only showing top 5 rows


In [21]:
words_clean = words_clean.filter(col('word') != '')
words_clean.show(5, truncate=False)

+----------+
|word      |
+----------+
|all       |
|songs     |
|considered|
|peter     |
|gabriel   |
+----------+
only showing top 5 rows


In [25]:
groups = words_clean.groupBy(col('word'))
counts = groups.count()
counts.orderBy(col('count').desc()).show(5, truncate=False)

+-----+-----+
|word |count|
+-----+-----+
|the  |9    |
|on   |5    |
|music|5    |
|and  |4    |
|a    |4    |
+-----+-----+
only showing top 5 rows


In [27]:
import pyspark.sql.functions as F

counts = (
    spark.read.text(file_path)
    .select(F.split(F.col('value'), ' ').alias('lines'))
    .select(F.explode(F.col('lines')).alias('word'))
    .select(F.lower(F.col('word')).alias('word'))
    .select(F.regexp_extract(F.col('word'), r'(\W+)?([a-z]+)', 2).alias('word'))
    .filter(F.col('word') != '')
    .groupBy(F.col('word'))
    .count()
    .orderBy(F.col('count').desc())
)

In [28]:
counts.show()

+-------+-----+
|   word|count|
+-------+-----+
|    the|    9|
|     on|    5|
|  music|    5|
|    and|    4|
|      a|    4|
|    pop|    3|
|    new|    3|
|     to|    3|
|  great|    3|
|     is|    3|
|     of|    3|
| divide|    2|
|   long|    2|
|    his|    2|
|   tiny|    2|
| singer|    2|
|michael|    2|
|    out|    2|
|   song|    2|
|   desk|    2|
+-------+-----+
only showing top 20 rows
